In [2]:
import numpy as np
from tqdm import tqdm
import scipy.sparse as sp
import scanpy as sc

def read_adata(path):
    adata = sc.read_h5ad(path)
    return adata
adata = read_adata('./only_hvg/Norman_only_hvg.h5ad')

In [3]:
adata.obs

,guide_id,read_count,UMI_count,coverage,gemgroup,good_coverage,number_of_cells,tissue_type,cell_line,cancer,...,perturbation_type,celltype,organism,perturbation,nperts,ngenes,ncounts,percent_mito,percent_ribo,n_genes
TTGAACGAGACTCGGA,ARID1A_NegCtrl0;ARID1A_NegCtrl0,28684,1809,15.856274,2,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,ARID1A,1,3079,15097.0,5.815725,33.569583,3079
CGTTGGGGTGTTTGTG,BCORL1_NegCtrl0;BCORL1_NegCtrl0,18367,896,20.498884,7,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,BCORL1,1,2100,8551.0,4.104783,45.842592,2100
GAACCTAAGTGTTAGA,FOSB_NegCtrl0;FOSB_NegCtrl0,16296,664,24.542169,6,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,FOSB,1,2772,10999.0,5.655060,17.801618,2772
CCTTCCCTCCGTCATC,SET_KLF1;SET_KLF1,16262,850,19.131765,4,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,SET_KLF1,2,5385,38454.0,4.335050,38.165080,5385
TCAATCTGTCTTTCAT,OSR2_NegCtrl0;OSR2_NegCtrl0,16057,1067,15.048735,2,True,2,cell_line,K562,True,...,CRISPR,lymphoblasts,human,OSR2,1,4869,27926.0,5.084867,32.317554,4869
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGCGCAGTCATGCT,RHOXF2_NegCtrl0;RHOXF2_NegCtrl0,1,1,1.000000,2,False,0,cell_line,K562,True,...,CRISPR,lymphoblasts,human,RHOXF2,1,1853,5192.0,5.508475,31.798921,1853
TTTGCGCCAGGACCCT,BCL2L11_BAK1;BCL2L11_BAK1,1,1,1.000000,3,False,0,cell_line,K562,True,...,CRISPR,lymphoblasts,human,BCL2L11_BAK1,2,3508,15704.0,6.718034,38.334182,3508
TTTGCGCGTACTTGAC-1,CNN1_NegCtrl0;CNN1_NegCtrl0,1,1,1.000000,3,False,0,cell_line,K562,True,...,CRISPR,lymphoblasts,human,CNN1,1,3609,15054.0,5.633054,29.440680,3609
TTTGCGCTCTCGCATC-1,CEBPB_OSR2;CEBPB_OSR2,1,1,1.000000,6,False,0,cell_line,K562,True,...,CRISPR,lymphoblasts,human,CEBPB_OSR2,2,2576,6825.0,2.695971,16.879121,2576


In [4]:
adata.X.max()

8.545323

In [ ]:
# 按cartessian_key划分细胞
import numpy as np
from tqdm import tqdm
import scipy.sparse as sp
import scanpy as sc

def read_adata(path):
    adata = sc.read_h5ad(path)
    return adata

def iter_adata_rows(
    adata,
    cols=["perturbation"],
    to_float32: bool = False,
):

    # 1) obs三列一次性取出，避免每行反复做 pandas 索引
    obs_df = adata.obs.loc[:, list(cols)]

    # 2) X：若是稀疏，尽量保证 CSR 以便按行切片更快
    # X = sp.csr_matrix(adata.X)
    
    X = sp.csr_matrix(adata.X)

    is_sparse = sp.issparse(X)

    for i in tqdm(range(adata.n_obs)):
        row_obs = obs_df.iloc[i]

        # 取obs字段
        out = {"i": i}
        for c in cols:
            out[c] = row_obs[c]
        

        out["x"] = X[i]
        yield out



group_dict = {}
# 用法示例




for item in iter_adata_rows(adata):
    perturbation, x =  item["perturbation"], item["x"]
    split_keys = perturbation.split("_")
    cell_key = (split_keys[0], split_keys[1]) if len(split_keys) > 1 else (split_keys[0], 'None')
    if cell_key not in group_dict:
        group_dict[cell_key] = []
    group_dict[cell_key].append(x)
    

    

100%|██████████| 111445/111445 [00:06<00:00, 18182.79it/s]


In [6]:
group_dict.keys()


dict_keys([('ARID1A', 'None'), ('BCORL1', 'None'), ('FOSB', 'None'), ('SET', 'KLF1'), ('OSR2', 'None'), ('KLF1', 'BAK1'), ('FOXA3', 'FOXL2'), ('TP73', 'None'), ('HES7', 'None'), ('IRF1', 'SET'), ('IGDCC3', 'MAPK1'), ('UBASH3B', 'None'), ('CBL', 'PTPN12'), ('MAP2K6', 'None'), ('SAMD1', 'PTPN12'), ('BCL2L11', 'None'), ('UBASH3A', 'None'), ('LYL1', 'IER5L'), ('CDKN1C', 'CDKN1A'), ('AHR', 'FEV'), ('LHX1', 'None'), ('FOXA3', 'None'), ('SGK1', 'TBX3'), ('C3orf72', 'FOXL2'), ('TGFBR2', 'PRTG'), ('CDKN1A', 'None'), ('HOXB9', 'None'), ('CDKN1C', 'CDKN1B'), ('control', 'None'), ('CSRNP1', 'None'), ('ISL2', 'None'), ('MAP2K3', 'MAP2K6'), ('CDKN1C', 'None'), ('MAP7D1', 'None'), ('CBL', 'TGFBR2'), ('MAPK1', 'IKZF3'), ('PTPN12', 'SNAI1'), ('LHX1', 'ELMSAN1'), ('FOXA3', 'FOXF1'), ('CDKN1B', 'None'), ('CEBPB', 'PTPN12'), ('PTPN1', 'None'), ('ZBTB25', 'None'), ('MAPK1', 'PRTG'), ('SET', 'None'), ('MAP2K3', 'IKZF3'), ('DUSP9', 'PRTG'), ('SLC4A1', 'None'), ('KLF1', 'COL2A1'), ('KLF1', 'CLDN6'), ('KLF1', 

In [7]:
for x in group_dict.keys():
    if 'control' in x:
        print(x)

('control', 'None')


In [8]:
key_list = list(group_dict.keys())
print(len(key_list))
len_list = np.array([len(group_dict[k]) for k in key_list], dtype=np.int64)
min(len_list)

237


54

In [9]:
import pickle
processed_cartesian_keys = list(group_dict.keys())
# 按1024 shard_size 进行配对的并保存
control_name = "control"
shard_size = 128
rng = np.random.default_rng(0)
pert_sample_list = []
control_dict = {}

def pad_for_shard_size(array, shard_size):
    N = array.shape[0]
    pad_n = (-N) % shard_size  # number of rows to add
    if pad_n == 0:
        return array, np.empty((0,), dtype=np.int64)
    pad_idx = rng.integers(0, N, size=pad_n)   # with replacement
    array_pad = sp.vstack([array, array[pad_idx]])
    return array_pad, pad_idx

def equal_pert_parts(pert_array, shard_size=1024):
    
    pert_array_pad, _ = pad_for_shard_size(pert_array, shard_size)
    pert_num = pert_array_pad.shape[0]

    
    for sample_idx, i in enumerate(range(0, pert_num, shard_size)):
        pert_sub = pert_array_pad[i:i+shard_size]
        sample = {
            "cartesian_key": pck,
            "cell_matrix": pert_sub,
        }
        pert_sample_list.append(sample)


for pck in tqdm(processed_cartesian_keys):
    if pck[0] == control_name:
        
        control_dict[pck] = sp.vstack(group_dict[pck], format="csr")
    else:
        pert_array = sp.vstack(group_dict[pck], format="csr")
        equal_pert_parts(pert_array, shard_size=shard_size)
    

len(pert_sample_list)

with open(f"pert_sample_list.pkl", "wb") as f:
    pickle.dump(pert_sample_list, f)

with open(f"control_dict.pkl", "wb") as f:
    pickle.dump(control_dict, f)

100%|██████████| 237/237 [00:00<00:00, 242.11it/s]
